In [1]:
import numpy as np
import math
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

# =========================================================
# 1. Data: 5D inputs and 1D outputs (Function 6)
# =========================================================

X_raw = np.array([
 [0.7281861, 0.15469257, 0.73255167, 0.69399651, 0.05640131],
 [0.24238435, 0.84409997, 0.5778091, 0.67902128, 0.50195289],
 [0.72952261, 0.7481062, 0.67977464, 0.35655228, 0.67105368],
 [0.77062024, 0.11440374, 0.04677993, 0.64832428, 0.27354905],
 [0.6188123, 0.33180214, 0.18728787, 0.75623847, 0.3288348],
 [0.78495809, 0.91068235, 0.7081201, 0.95922543, 0.0049115],
 [0.14511079, 0.8966846, 0.89632223, 0.72627154, 0.23627199],
 [0.94506907, 0.28845905, 0.97880576, 0.96165559, 0.59801594],
 [0.12572016, 0.86272469, 0.02854433, 0.24660527, 0.75120624],
 [0.75759436, 0.35583141, 0.0165229, 0.4342072, 0.11243304],
 [0.5367969, 0.30878091, 0.41187929, 0.38822518, 0.5225283],
 [0.95773967, 0.23566857, 0.09914585, 0.15680593, 0.07131737],
 [0.6293079, 0.80348368, 0.81140844, 0.04561319, 0.11062446],
 [0.02173531, 0.42808424, 0.83593944, 0.48948866, 0.51108173],
 [0.43934426, 0.69892383, 0.42682022, 0.10947609, 0.87788847],
 [0.25890557, 0.79367771, 0.6421139, 0.19667346, 0.59310318],
 [0.43216593, 0.71561781, 0.3418191, 0.70499988, 0.61496184],
 [0.78287982, 0.53633586, 0.44328356, 0.85969983, 0.01032599],
 [0.9217762, 0.93187122, 0.41487637, 0.59505727, 0.73562569],
 [0.12667892, 0.2914703, 0.06452848, 0.6805146, 0.89281919],
 [1.057739, 1.031871, 1.078805, 1.061655, 0.992819],
 [0.183405, 0.304243, 0.524756, 0.431945, 0.29123 ],
 [0.268807, 0.268756, 0.495982, 0.986904, 0.010463],
 [0.071886, 0.119564, 0.11427 , 0.97486 , 0.062381],
 [0.985053, 0.912856, 0.967723, 0.993021, 0.944062],
 [0.985053, 0.912856, 0.967723, 0.993021, 0.944062]
], dtype=float)

# Make y_raw a clean 1D vector (length must match X_raw rows)
y_raw = np.array([
 -0.71426495, -1.20995524, -1.67219994, -1.53605771, -0.82923655,
 -1.24704893, -1.23378638, -1.69434344, -2.57116963, -1.30911635,
 -1.14478485, -1.91267714, -1.62283895, -1.35668211, -2.0184254,
 -1.70255784, -1.29424696, -0.93575656, -2.15576776, -1.74688209,
 -2.868905011263093, -0.9489046340640067, -0.6674108573004914,
 -1.3769650251311083, -2.5104529076756172, -2.5498833751068073
], dtype=float)

assert X_raw.shape[0] == y_raw.shape[0], "X and y must have the same number of rows."

# =========================================================
# 2. Device & Seeds
# =========================================================

RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# 3. Scaling
# =========================================================

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_scaled = x_scaler.fit_transform(X_raw)
y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

X_tensor_all = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor_all = torch.tensor(y_scaled.reshape(-1, 1), dtype=torch.float32, device=device)

# =========================================================
# 4. Surrogate Network (configurable)
# =========================================================

class SurrogateNN(nn.Module):
    def __init__(self, input_dim=5, hidden_layers=(128, 128), dropout_p=0.1):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_layers:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_p))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# =========================================================
# 5. Train helper (early stopping)
# =========================================================

def train_with_early_stopping(model, optimizer, criterion, X_train, y_train, X_val, y_val,
                              max_epochs=2000, patience=250):
    best_state = None
    best_val = float("inf")
    no_improve = 0

    for _ in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(X_train)
        loss = criterion(pred, y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = criterion(val_pred, y_val).item()

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    return best_val

# =========================================================
# 6. Hyperparameter tuning (Random Search + CV)
# =========================================================

def sample_config(rng):
    hidden_choices = [(64, 64), (128, 64), (128, 128), (256, 128)]
    dropout_choices = [0.0, 0.05, 0.1, 0.2, 0.3]
    lr_choices = [5e-4, 1e-3, 2e-3, 3e-3]
    wd_choices = [0.0, 1e-6, 1e-5, 1e-4, 2e-4]

    return {
        "hidden_layers": hidden_choices[rng.integers(0, len(hidden_choices))],
        "dropout": float(dropout_choices[rng.integers(0, len(dropout_choices))]),
        "lr": float(lr_choices[rng.integers(0, len(lr_choices))]),
        "weight_decay": float(wd_choices[rng.integers(0, len(wd_choices))]),
        "max_epochs": int(rng.choice([800, 1200, 1600, 2000])),
        "patience": int(rng.choice([150, 200, 250, 300])),
        "mc_samples": int(rng.choice([50, 80, 100, 150])),
        "xi_base": float(rng.choice([0.0, 0.005, 0.01, 0.02]))
    }

def cv_score_config(cfg, X_all, y_all, seed=123, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    criterion = nn.MSELoss()
    vals = []

    for tr_idx, va_idx in kf.split(X_all):
        X_tr = X_all[tr_idx]
        y_tr = y_all[tr_idx]
        X_va = X_all[va_idx]
        y_va = y_all[va_idx]

        model = SurrogateNN(input_dim=5, hidden_layers=cfg["hidden_layers"], dropout_p=cfg["dropout"]).to(device)
        optimizer = optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

        val = train_with_early_stopping(
            model, optimizer, criterion,
            X_tr, y_tr, X_va, y_va,
            max_epochs=cfg["max_epochs"], patience=cfg["patience"]
        )
        vals.append(val)

    return float(np.mean(vals))

def tune_hyperparameters(X_all, y_all, n_trials=25, seed=123):
    rng = np.random.default_rng(seed)
    best_cfg = None
    best_score = float("inf")

    for t in range(1, n_trials + 1):
        cfg = sample_config(rng)
        score = cv_score_config(cfg, X_all, y_all, seed=seed, n_splits=5)

        if score < best_score:
            best_score = score
            best_cfg = cfg

        print(f"Trial {t:02d}/{n_trials} | CV-MSE={score:.6f} | cfg={cfg}")

    return best_cfg, best_score

# =========================================================
# 7. MC Dropout Prediction
# =========================================================

def predict_mc(model, X_candidates, mc_samples=100):
    model.train()  # keep dropout active
    X_scaled_cand = x_scaler.transform(X_candidates)
    X_t = torch.tensor(X_scaled_cand, dtype=torch.float32, device=device)

    preds = []
    with torch.no_grad():
        for _ in range(mc_samples):
            preds.append(model(X_t).cpu().numpy())

    preds = np.array(preds).squeeze(-1)  # shape: [mc, n]

    # invert scaling
    preds_flat = preds.reshape(-1, 1)
    preds_orig = y_scaler.inverse_transform(preds_flat).reshape(preds.shape)

    mu = preds_orig.mean(axis=0)
    sigma = preds_orig.std(axis=0)
    return mu, sigma

# =========================================================
# 8. Acquisition (EI & PI)
# =========================================================

def normal_pdf(z):
    return np.exp(-0.5 * z**2) / math.sqrt(2 * math.pi)

def normal_cdf(z):
    z = np.asarray(z)
    return 0.5 * (1 + np.vectorize(math.erf)(z / math.sqrt(2)))

def acquisition_ei_pi(mu, sigma, y_best, xi=0.0):
    eps = 1e-9
    sigma = np.maximum(sigma, eps)

    improvement = mu - y_best - xi
    Z = improvement / sigma

    cdf_vals = normal_cdf(Z)
    pdf_vals = normal_pdf(Z)

    ei = improvement * cdf_vals + sigma * pdf_vals
    pi = cdf_vals

    return np.maximum(ei, 0.0), pi

# =========================================================
# 9. Propose Next Point (Hybrid EI/PI + non-duplicate)
# =========================================================

def propose_next_point(model, X_obs, y_obs, mc_samples=100, xi_base=0.01,
                       n_candidates=30000, random_seed=123, dup_tol=1e-6,
                       ei_top_frac=0.05):
    rng = np.random.default_rng(random_seed)
    X_cand = rng.random((n_candidates, 5))

    # remove duplicates (exact / near-exact)
    # simple filter: drop candidates extremely close to any observed point
    keep = np.ones(n_candidates, dtype=bool)
    for i in range(n_candidates):
        if not keep[i]:
            continue
        if np.any(np.linalg.norm(X_obs - X_cand[i], axis=1) < dup_tol):
            keep[i] = False
    X_cand = X_cand[keep]

    mu, sigma = predict_mc(model, X_cand, mc_samples=mc_samples)

    y_best = float(np.max(y_obs))
    y_min = float(np.min(y_obs))
    y_range = max(y_best - y_min, 1e-6)
    effective_xi = xi_base * y_range

    ei, pi = acquisition_ei_pi(mu, sigma, y_best, xi=effective_xi)

    # Hybrid policy:
    # 1) find top EI region (within ei_top_frac of max EI)
    # 2) inside that region, choose max PI (more reliable improvement probability)
    max_ei = float(np.max(ei))
    thresh = max_ei * (1.0 - ei_top_frac)
    idx_pool = np.where(ei >= thresh)[0]

    if len(idx_pool) == 0:
        best_idx = int(np.argmax(ei))
    else:
        best_idx = int(idx_pool[np.argmax(pi[idx_pool])])

    return {
        "x_next": X_cand[best_idx],
        "mu_next": float(mu[best_idx]),
        "sigma_next": float(sigma[best_idx]),
        "pi_next": float(pi[best_idx]),
        "ei_next": float(ei[best_idx]),
        "x_best_obs": X_obs[int(np.argmax(y_obs))],
        "y_best_obs": float(y_best),
        "max_ei": max_ei,
        "ei_threshold": float(thresh),
        "pool_size": int(len(idx_pool))
    }

# =========================================================
# 10. Local Sensitivity (Gradients)
# =========================================================

def local_sensitivity(model, x_point):
    model.eval()
    x_scaled = x_scaler.transform(x_point.reshape(1, -1))
    x_t = torch.tensor(x_scaled, dtype=torch.float32, device=device, requires_grad=True)

    y_pred = model(x_t)
    y_pred.backward()

    grads = x_t.grad.detach().cpu().numpy().flatten()
    g = np.abs(grads)
    if g.sum() == 0:
        return np.ones_like(g) / len(g)
    return g / g.sum()

# =========================================================
# 11. Main (Tune -> Train best -> Propose)
# =========================================================

def main():
    print("\n================ WEEK 7 FUNCTION 6 — HYPERPARAMETER TUNING ================\n")

    best_cfg, best_cv = tune_hyperparameters(
        X_tensor_all, y_tensor_all,
        n_trials=25,
        seed=RANDOM_SEED
    )

    print("\n================ BEST TUNED CONFIG ================\n")
    print("Best CV-MSE (scaled y):", best_cv)
    print("Best config:", best_cfg)

    # Train final model on ALL data using best config.
    final_model = SurrogateNN(input_dim=5, hidden_layers=best_cfg["hidden_layers"], dropout_p=best_cfg["dropout"]).to(device)
    optimizer = optim.Adam(final_model.parameters(), lr=best_cfg["lr"], weight_decay=best_cfg["weight_decay"])
    criterion = nn.MSELoss()

    # Use a small validation split for early stopping on full training
    n = X_tensor_all.shape[0]
    idx = torch.randperm(n)
    n_train = max(int(0.85 * n), 1)
    tr_idx, va_idx = idx[:n_train], idx[n_train:]

    train_with_early_stopping(
        final_model, optimizer, criterion,
        X_tensor_all[tr_idx], y_tensor_all[tr_idx],
        X_tensor_all[va_idx], y_tensor_all[va_idx],
        max_epochs=best_cfg["max_epochs"],
        patience=best_cfg["patience"]
    )

    details = propose_next_point(
        final_model,
        X_raw,
        y_raw,
        mc_samples=best_cfg["mc_samples"],
        xi_base=best_cfg["xi_base"],
        n_candidates=30000,
        random_seed=RANDOM_SEED,
        dup_tol=1e-6,
        ei_top_frac=0.05
    )

    print("\n================ CURRENT BEST OBSERVED ================\n")
    print("x_best =", details["x_best_obs"])
    print("y_best =", details["y_best_obs"])

    print("\n================ SELECTION POLICY (Hybrid EI/PI) ================\n")
    print(f"- max_EI = {details['max_ei']:.6f}")
    print(f"- EI threshold (top {int(100*0.05)}%) = {details['ei_threshold']:.6f}")
    print(f"- Candidate pool size above threshold = {details['pool_size']}")

    print("\n================ RECOMMENDED NEXT POINT ================\n")
    print("x_next =", details["x_next"])
    print("mu(x_next) =", details["mu_next"])
    print("sigma(x_next) =", details["sigma_next"])
    print("EI =", details["ei_next"])
    print("PI =", details["pi_next"])

    print("\n================ LOCAL SENSITIVITY ================\n")
    sens = local_sensitivity(final_model, details["x_next"])
    for i, s in enumerate(sens, 1):
        print(f"Dim {i}: {s:.3f}")

if __name__ == "__main__":
    main()



================ WEEK 7 FUNCTION 6 — HYPERPARAMETER TUNING ================

Trial 01/25 | CV-MSE=0.253262 | cfg={'hidden_layers': (64, 64), 'dropout': 0.2, 'lr': 0.002, 'weight_decay': 0.0, 'max_epochs': 2000, 'patience': 150, 'mc_samples': 80, 'xi_base': 0.0}
Trial 02/25 | CV-MSE=0.303970 | cfg={'hidden_layers': (128, 64), 'dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.0002, 'max_epochs': 1200, 'patience': 300, 'mc_samples': 80, 'xi_base': 0.005}
Trial 03/25 | CV-MSE=0.210314 | cfg={'hidden_layers': (256, 128), 'dropout': 0.3, 'lr': 0.003, 'weight_decay': 0.0002, 'max_epochs': 800, 'patience': 250, 'mc_samples': 80, 'xi_base': 0.0}
Trial 04/25 | CV-MSE=0.218976 | cfg={'hidden_layers': (64, 64), 'dropout': 0.3, 'lr': 0.003, 'weight_decay': 1e-06, 'max_epochs': 1200, 'patience': 250, 'mc_samples': 50, 'xi_base': 0.01}
Trial 05/25 | CV-MSE=0.218050 | cfg={'hidden_layers': (128, 64), 'dropout': 0.3, 'lr': 0.002, 'weight_decay': 1e-06, 'max_epochs': 2000, 'patience': 300, 'mc_samples': 5